# Notebook 3: Modeling — Can We Predict Where a Gym Will Thrive?
**Project:** Where Do Fitness Businesses Thrive in Utah?

We run **two supervised tasks** and one **unsupervised** task:

| Task | Type | Target | Question |
|---|---|---|---|
| A | Binary Classifier | `is_thriving` | Will this business survive and perform well? |
| B | Regressor | `success_score` | How successful will an open business be? |
| C | Clustering | — | What market segments exist across Utah zip codes? |

All models use an 80/20 train/test split with 5-fold CV for tuning.
All preprocessing lives inside `sklearn` Pipelines to prevent data leakage.

## 0. Imports & Setup

In [ ]:
NUMERIC_FEATURES = [
    'median_income', 'median_home_value', 'median_age',
    'pct_bachelors', 'pct_prime_gym_age', 'total_pop',
    'competition_1km', 'competition_3km',
    'market_gap', 'gym_density_per_1k', 'income_per_competitor',
    'price',
]
CATEGORICAL_FEATURES = ['category_group']
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES),
])

print('Preprocessor configured.')

In [ ]:
# ── Shared preprocessing config ────────────────────────────────────────────────
NUMERIC_FEATURES = [
    'median_income', 'median_home_value', 'median_age',
    'pct_bachelors', 'pct_prime_gym_age', 'total_pop',
    'competition_1km', 'competition_3km',
    'market_gap', 'gym_density_per_1k', 'income_per_competitor',
    'price', 'has_hours',
]
CATEGORICAL_FEATURES = ['category_group']
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES),
])

print('Preprocessor configured.')

---
## Task A: Binary Classification — Will This Business Thrive?

In [ ]:
X_clf = clf_df[ALL_FEATURES]
y_clf = clf_df['is_thriving'].astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
print(f'Train: {len(X_train_c)} | Test: {len(X_test_c)}')
print(f'Class balance (train): {y_train_c.mean():.2f} thriving')

### A1. Logistic Regression (Baseline)

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

lr_params = {'model__C': [0.01, 0.1, 1.0, 10.0]}
lr_grid = GridSearchCV(lr_pipeline, lr_params, cv=5, scoring='roc_auc', n_jobs=-1)
lr_grid.fit(X_train_c, y_train_c)

lr_best = lr_grid.best_estimator_
y_pred_lr   = lr_best.predict(X_test_c)
y_proba_lr  = lr_best.predict_proba(X_test_c)[:, 1]

print('Best params:', lr_grid.best_params_)
print(f'Accuracy: {accuracy_score(y_test_c, y_pred_lr):.3f}')
print(f'ROC-AUC:  {roc_auc_score(y_test_c, y_proba_lr):.3f}')
print(classification_report(y_test_c, y_pred_lr, target_names=['Not Thriving', 'Thriving']))

### A2. Random Forest Classifier

In [ ]:
rf_clf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42, class_weight='balanced')),
])

rf_clf_params = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5],
}
rf_clf_grid = GridSearchCV(rf_clf_pipeline, rf_clf_params, cv=5, scoring='roc_auc', n_jobs=-1)
rf_clf_grid.fit(X_train_c, y_train_c)

rf_clf_best = rf_clf_grid.best_estimator_
y_pred_rf_c  = rf_clf_best.predict(X_test_c)
y_proba_rf_c = rf_clf_best.predict_proba(X_test_c)[:, 1]

print('Best params:', rf_clf_grid.best_params_)
print(f'Accuracy: {accuracy_score(y_test_c, y_pred_rf_c):.3f}')
print(f'ROC-AUC:  {roc_auc_score(y_test_c, y_proba_rf_c):.3f}')
print(classification_report(y_test_c, y_pred_rf_c, target_names=['Not Thriving', 'Thriving']))

### A3. Gradient Boosting Classifier

In [ ]:
gb_clf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(random_state=42)),
])

gb_clf_params = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [3, 5],
}
gb_clf_grid = GridSearchCV(gb_clf_pipeline, gb_clf_params, cv=5, scoring='roc_auc', n_jobs=-1)
gb_clf_grid.fit(X_train_c, y_train_c)

gb_clf_best = gb_clf_grid.best_estimator_
y_pred_gb_c  = gb_clf_best.predict(X_test_c)
y_proba_gb_c = gb_clf_best.predict_proba(X_test_c)[:, 1]

print('Best params:', gb_clf_grid.best_params_)
print(f'Accuracy: {accuracy_score(y_test_c, y_pred_gb_c):.3f}')
print(f'ROC-AUC:  {roc_auc_score(y_test_c, y_proba_gb_c):.3f}')
print(classification_report(y_test_c, y_pred_gb_c, target_names=['Not Thriving', 'Thriving']))

### A4. Classifier Comparison — ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for name, proba in [
    ('Logistic Regression', y_proba_lr),
    ('Random Forest', y_proba_rf_c),
    ('Gradient Boosting', y_proba_gb_c),
]:
    fpr, tpr, _ = roc_curve(y_test_c, proba)
    auc = roc_auc_score(y_test_c, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Is This Business Thriving?')
ax.legend()
plt.tight_layout()
plt.savefig('figures/03a_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### A5. Feature Importance (Best Classifier)

In [ ]:
# Use whichever classifier had the best AUC (update model variable if needed)
best_clf_model = gb_clf_best  # change to rf_clf_best if RF wins

ohe_features = (
    best_clf_model.named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['onehot']
    .get_feature_names_out(CATEGORICAL_FEATURES)
).tolist()
all_feature_names = NUMERIC_FEATURES + ohe_features

importances = best_clf_model.named_steps['model'].feature_importances_
fi_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color='steelblue')
ax.set_title('Top 15 Feature Importances (Classifier)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('figures/03a_feature_importance_classifier.png', dpi=150, bbox_inches='tight')
plt.show()

print(fi_df.to_string(index=False))

---
## Task B: Regression — How Successful Will an Open Business Be?

In [ ]:
X_reg = reg_df[ALL_FEATURES]
y_reg = reg_df['success_score']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train_r)} | Test: {len(X_test_r)}')

### B1. Ridge Regression (Baseline)

In [ ]:
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge()),
])
ridge_params = {'model__alpha': [0.1, 1.0, 10.0, 100.0]}
ridge_grid = GridSearchCV(ridge_pipeline, ridge_params, cv=5,
                          scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_grid.fit(X_train_r, y_train_r)

ridge_best = ridge_grid.best_estimator_
y_pred_ridge = ridge_best.predict(X_test_r)

print('Best params:', ridge_grid.best_params_)
print(f'RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_ridge)):.4f}')
print(f'R²:   {r2_score(y_test_r, y_pred_ridge):.4f}')

### B2. Random Forest Regressor

In [ ]:
rf_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42)),
])
rf_reg_params = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5],
}
rf_reg_grid = GridSearchCV(rf_reg_pipeline, rf_reg_params, cv=5,
                           scoring='neg_root_mean_squared_error', n_jobs=-1)
rf_reg_grid.fit(X_train_r, y_train_r)

rf_reg_best = rf_reg_grid.best_estimator_
y_pred_rf_r = rf_reg_best.predict(X_test_r)

print('Best params:', rf_reg_grid.best_params_)
print(f'RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_rf_r)):.4f}')
print(f'R²:   {r2_score(y_test_r, y_pred_rf_r):.4f}')

### B3. Gradient Boosting Regressor

In [ ]:
gb_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42)),
])
gb_reg_params = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [3, 5],
}
gb_reg_grid = GridSearchCV(gb_reg_pipeline, gb_reg_params, cv=5,
                           scoring='neg_root_mean_squared_error', n_jobs=-1)
gb_reg_grid.fit(X_train_r, y_train_r)

gb_reg_best = gb_reg_grid.best_estimator_
y_pred_gb_r = gb_reg_best.predict(X_test_r)

print('Best params:', gb_reg_grid.best_params_)
print(f'RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_gb_r)):.4f}')
print(f'R²:   {r2_score(y_test_r, y_pred_gb_r):.4f}')

### B4. Regression Model Comparison

In [ ]:
results = pd.DataFrame([
    {'Model': 'Ridge',              'RMSE': np.sqrt(mean_squared_error(y_test_r, y_pred_ridge)), 'R²': r2_score(y_test_r, y_pred_ridge)},
    {'Model': 'Random Forest',      'RMSE': np.sqrt(mean_squared_error(y_test_r, y_pred_rf_r)),  'R²': r2_score(y_test_r, y_pred_rf_r)},
    {'Model': 'Gradient Boosting',  'RMSE': np.sqrt(mean_squared_error(y_test_r, y_pred_gb_r)),  'R²': r2_score(y_test_r, y_pred_gb_r)},
]).round(4)

print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results.plot.bar(x='Model', y='RMSE', ax=axes[0], color='tomato', legend=False)
axes[0].set_title('RMSE (lower = better)')
axes[0].set_xticklabels(results['Model'], rotation=20, ha='right')

results.plot.bar(x='Model', y='R²', ax=axes[1], color='steelblue', legend=False)
axes[1].set_title('R² (higher = better)')
axes[1].set_xticklabels(results['Model'], rotation=20, ha='right')

plt.tight_layout()
plt.savefig('figures/03b_regression_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Task C: K-Means Clustering — Market Segmentation by Zip Code

We cluster Utah zip codes by their market characteristics to identify distinct segments:
- High-income, low-competition opportunity zones
- Dense urban, high-competition markets
- Student-heavy, budget-oriented areas
- etc.

Then we overlay the classifier's `is_thriving` predictions onto the map.

In [ ]:
# Build zip-level features for clustering
full_df = pd.read_csv('data/utah_fitness_v2.csv', dtype={'zip_code': str})

zip_features = full_df.groupby('zip_code').agg(
    median_income=('median_income', 'first'),
    median_home_value=('median_home_value', 'first'),
    pct_prime_gym_age=('pct_prime_gym_age', 'first'),
    pct_bachelors=('pct_bachelors', 'first'),
    total_pop=('total_pop', 'first'),
    gyms_in_zip=('gyms_in_zip', 'first'),
    market_gap=('market_gap', 'first'),
    avg_rating=('rating', lambda x: x[x > 0].mean()),
    pct_closed=('is_closed', 'mean'),
    lat=('latitude', 'mean'),
    lon=('longitude', 'mean'),
).dropna().reset_index()

CLUSTER_FEATURES = [
    'median_income', 'median_home_value', 'pct_prime_gym_age',
    'pct_bachelors', 'total_pop', 'gyms_in_zip', 'market_gap',
    'avg_rating', 'pct_closed',
]

scaler = StandardScaler()
X_clust = scaler.fit_transform(zip_features[CLUSTER_FEATURES].fillna(0))

print(f'Clustering {len(zip_features)} zip codes.')

In [ ]:
# Elbow method to choose K
inertias = []
sil_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clust)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_clust, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_range), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(list(K_range), sil_scores, 'o-', color='tomato')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')

plt.tight_layout()
plt.savefig('figures/03c_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

print('Silhouette scores:', {k: round(s, 3) for k, s in zip(K_range, sil_scores)})

In [ ]:
# Fit final K-means (adjust K based on elbow/silhouette above)
BEST_K = 4  # update after reviewing the plots above

km_final = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
zip_features['cluster'] = km_final.fit_predict(X_clust)

# Cluster profiles
cluster_profile = zip_features.groupby('cluster')[CLUSTER_FEATURES].mean().round(2)
print('Cluster profiles:')
print(cluster_profile.T)

In [ ]:
# Visualize clusters on a Utah scatter map (lat/lon)
fig, ax = plt.subplots(figsize=(8, 10))

colors = ['steelblue', 'tomato', 'seagreen', 'orange', 'purple', 'brown']
cluster_labels = {
    0: 'Cluster 0',  # update with descriptive names after reviewing profiles
    1: 'Cluster 1',
    2: 'Cluster 2',
    3: 'Cluster 3',
}

for cluster_id in sorted(zip_features['cluster'].unique()):
    subset = zip_features[zip_features['cluster'] == cluster_id]
    ax.scatter(
        subset['lon'], subset['lat'],
        c=colors[cluster_id],
        label=cluster_labels.get(cluster_id, f'Cluster {cluster_id}'),
        s=subset['total_pop'] / 500,  # size by population
        alpha=0.7, edgecolors='white', linewidths=0.5
    )

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Utah Fitness Market Segments by Zip Code\n(point size = population)')
ax.legend()
plt.tight_layout()
plt.savefig('figures/03c_utah_clusters_map.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Merge cluster labels back and look at thriving rate per cluster
full_df = full_df.merge(zip_features[['zip_code', 'cluster']], on='zip_code', how='left')

thriving_by_cluster = (
    full_df[full_df['is_thriving'].notna()]
    .groupby('cluster')['is_thriving']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'pct_thriving', 'count': 'n_businesses'})
    .round(3)
)

print('Thriving rate by market segment:')
print(thriving_by_cluster)

---
## Final Summary Table

In [ ]:
# Fill in after running — update AUC and R² values from above
summary = pd.DataFrame([
    {'Task': 'A - Classifier', 'Model': 'Logistic Regression',     'Metric': 'ROC-AUC', 'Score': '???'},
    {'Task': 'A - Classifier', 'Model': 'Random Forest',           'Metric': 'ROC-AUC', 'Score': '???'},
    {'Task': 'A - Classifier', 'Model': 'Gradient Boosting',       'Metric': 'ROC-AUC', 'Score': '???'},
    {'Task': 'B - Regressor',  'Model': 'Ridge',                   'Metric': 'R²',      'Score': '???'},
    {'Task': 'B - Regressor',  'Model': 'Random Forest',           'Metric': 'R²',      'Score': '???'},
    {'Task': 'B - Regressor',  'Model': 'Gradient Boosting',       'Metric': 'R²',      'Score': '???'},
    {'Task': 'C - Clustering', 'Model': f'K-Means (K={BEST_K})',   'Metric': 'Silhouette', 'Score': '???'},
])
print(summary.to_string(index=False))